# Weather Risk Forecasting for Energy Operations

This notebook walks through a complete applied ML workflow for classifying daily weather conditions as safe or unsafe for outdoor energy operations.

## 1. Problem framing

The operational question is: given daily weather variables, can we flag days that may be unsafe for outdoor energy operations? The target is binary: `safe` or `unsafe`. Unsafe days are defined using transparent thresholds for wind, gusts, rainfall and temperature.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from weather_risk.data import generate_synthetic_weather_data
from weather_risk.features import prepare_modeling_table, get_feature_columns
from weather_risk.modeling import train_and_evaluate, select_best_model, time_aware_train_test_split
from weather_risk.visualization import save_confusion_matrix, save_feature_importance

## 2. Generate or load data

The first version uses synthetic weather data with seasonality, random variability, missing values and injected compound extreme events. This keeps the repository reproducible while preserving a realistic industrial analytics narrative.

In [ ]:
raw_df = generate_synthetic_weather_data(n_days=1460, start_date="2020-01-01")
raw_df.head()

In [ ]:
raw_df.describe().T

## 3. Clean data, define target and engineer features

The target is generated from operational thresholds. In a real project, this target could be replaced by historical work stoppages, HSE events, site access restrictions or manually validated unsafe days.

In [ ]:
df = prepare_modeling_table(raw_df)
df[["date", "wind_speed_max_kmh", "wind_gust_max_kmh", "precipitation_mm", "temperature_min_c", "temperature_max_c", "risk_label"]].head()

In [ ]:
df["risk_label"].value_counts(normalize=True).rename("share")

## 4. Visualise operational risk patterns

In [ ]:
ax = df["risk_label"].value_counts().reindex(["safe", "unsafe"]).plot(kind="bar", figsize=(7,4))
ax.set_title("Operational Weather Risk Distribution")
ax.set_xlabel("Risk class")
ax.set_ylabel("Number of days")
plt.tight_layout()

In [ ]:
monthly = df.set_index("date").resample("ME")["is_unsafe"].mean()
ax = monthly.plot(figsize=(10,4))
ax.set_title("Monthly Share of Unsafe Days")
ax.set_xlabel("Date")
ax.set_ylabel("Unsafe day rate")
plt.tight_layout()

## 5. Train supervised classification models

A time-aware split is used: earlier dates are used for training and later dates for testing. This is more realistic than a random split for weather and operational data.

In [ ]:
models, metrics = train_and_evaluate(df)
metrics_df = pd.DataFrame(metrics).T[["accuracy", "precision", "recall", "f1"]]
metrics_df

In [ ]:
best_name = select_best_model(metrics, criterion="f1")
best_model = models[best_name]
best_name

## 6. Interpret model results

For this use case, recall for the unsafe class is particularly important: missing an unsafe day can have safety, cost and scheduling consequences.

In [ ]:
ax = metrics_df.plot(kind="bar", figsize=(9,5))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
plt.tight_layout()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
X_train, X_test, y_train, y_test = time_aware_train_test_split(df)
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, display_labels=["safe", "unsafe"], colorbar=False)
plt.title(f"Confusion Matrix - {best_name}")
plt.tight_layout()

In [ ]:
estimator = best_model.named_steps["model"] if hasattr(best_model, "named_steps") else best_model
if hasattr(estimator, "feature_importances_"):
    importance = pd.Series(estimator.feature_importances_, index=get_feature_columns()).sort_values().tail(12)
    ax = importance.plot(kind="barh", figsize=(8,5))
    ax.set_title(f"Top Feature Importances - {best_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()

## 7. Operational interpretation

The model learns to prioritise weather variables directly linked to operational risk: wind gusts, maximum wind speed, precipitation and temperature extremes. In a production environment, this workflow could support a daily go / no-go planning dashboard or integrate with project scheduling systems.

## 8. Next steps

- Replace synthetic data with real station or forecast data.
- Add location-specific thresholds.
- Calibrate the probability threshold using operational cost assumptions.
- Add model monitoring and drift checks.
- Connect the pipeline to a Streamlit decision-support interface.